# Training a Small Language Model (TinyGPT) on a Technical QA Dataset

This notebook performs hyperparameter search for a lightweight Transformer model (`TinyGPT`) and evaluates it on a QA dataset generated from a product manual. The goal is to find the best trade-off between model size (footprint on embedded devices) and performance (loss, perplexity, BLEU, exact match).

## Notebook Structure
1. **Imports and setup** – load required libraries and custom modules.
2. **Load and split the QA dataset** – use `QADataset` to load JSONL data.
3. **Hyperparameter search loop** – train many model configurations and collect metrics.
4. **Evaluation and analysis** – generate summary CSV, Pareto frontier, parallel coordinates plots, and select the best model.

All code is kept as in the original; only comments and printed messages have been translated to English, and explanatory markdown cells have been added.

In [1]:
#!pip install -r requirements.txt

In [2]:
from torch.optim import AdamW
from torch.nn import CrossEntropyLoss
import os


import re, unicodedata
from collections import Counter
from torch.utils.data import DataLoader
import random
import pandas as pd
from sklearn.model_selection import train_test_split

from src.slm import TinyGPT
from src.utils.dataset import QADataset
from src.utils.tokenizer import CustomTokenizer


In [3]:
random.seed(0)
DATASET_PATH = "./data/dataset_CAT-4083-UK.json"
VOCAB_SIZE   = 1000

def preprocess_text(text):
    text = str(text).strip().lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(c for c in text if not unicodedata.combining(c))
    text = re.sub(r"[^a-z0-9\s.,!?'\-/:]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

if not os.path.exists(DATASET_PATH):
    print("Gerando dataset sintético...")
    topics = ["temperature","pressure","volume","mass","energy",
              "velocity","acceleration","force"]
    generic_qna = []
    for _ in range(1000):
        t = random.choice(topics)
        q = f"What is the unit of {t}"
        a = f"The unit of {t} is " + random.choice(
            ["kg","m/s","J","N","Pa","K","m3","W"])
        generic_qna.append({"question": preprocess_text(q),
                            "answer":   preprocess_text(a)})
else:
    df = pd.read_json(DATASET_PATH, lines=True)
    print(f"Total de registros carregados: {len(df)}")
    texts = []
    for _, row in df.iterrows():
        q = preprocess_text(row["question"]); a = preprocess_text(row["answer"])
        if q: texts.append(q)
        if a: texts.append(a)
    counter = Counter()
    for t in texts:
        counter.update(t.split())
    vocab = [w for w, _ in counter.most_common(VOCAB_SIZE)]
    vocab_set = set(vocab)
    generic_qna = []
    for _, row in df.iterrows():
        q = [w for w in preprocess_text(row["question"]).split() if w in vocab_set]
        a = [w for w in preprocess_text(row["answer"]).split() if w in vocab_set]
        if q and a:
            generic_qna.append({"question": " ".join(q), "answer": " ".join(a)})

print(f"Total de pares Q&A: {len(generic_qna)}")
for item in generic_qna[:3]:
    print("Q:", item["question"]); print("A:", item["answer"]); print()

Total de registros carregados: 132
Total de pares Q&A: 132
Q: what standards do the instruments meet according to the eu?
A: din/en 61000-6-2 and din/en 61000-6-3.

Q: how many pages are dedicated to pressure sensor selection guides?
A: two pages 6 and 7 .

Q: what pressure ranges do scp09 and scp10 sensors measure?
A: scp09 and scp10 measure mid and high pressure ranges.



In [4]:
MAX_WORDS = 30

filtered_qna = [
    item for item in generic_qna
    if len(item["question"].split()) <= MAX_WORDS
    and len(item["answer"].split()) <= MAX_WORDS
]

print(f"Dataset original: {len(generic_qna)}")
print(f"Dataset filtrado: {len(filtered_qna)}")
print(f"Removidos: {len(generic_qna) - len(filtered_qna)}")

Dataset original: 132
Dataset filtrado: 132
Removidos: 0


## 2. Load and Prepare the QA Dataset
The dataset was generated in the previous notebook. We load it from a JSONL file and split into train/test sets (80/20).

In [5]:
n = 20  # ou min(5000, len(filtered_qna))

train_qna_full = filtered_qna[:n]

tokenizer = CustomTokenizer(data_source=filtered_qna)

MAX_LEN = 50

split = int(len(train_qna_full) * 0.7)

train_qna = train_qna_full[:split]
eval_qna = train_qna_full[split:]

train_dataset = QADataset(
    qna_list=train_qna,
    tokenizer=tokenizer,
    max_length=MAX_LEN
)

eval_dataset = QADataset(
    qna_list=eval_qna,
    tokenizer=tokenizer,
    max_length=MAX_LEN
)

loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True
)

eval_loader = DataLoader(
    eval_dataset,
    batch_size=8,
    shuffle=False
)

xb, yb = next(iter(loader))

print(
    "vocab_size:", tokenizer.vocab_size,
    "| xb:", xb.shape,
    "| yb:", yb.shape
)

topics = [""] * len(eval_qna)

print(
    "train:", len(train_qna),
    "| eval:", len(eval_qna)
)

[INFO] Vocabulary built from Q&A with 489 tokens.
[INFO] Max original sequence length found: 22
[INFO] Sequences adjusted to max_length: 50
[INFO] Max original sequence length found: 20
[INFO] Sequences adjusted to max_length: 50
vocab_size: 489 | xb: torch.Size([8, 50]) | yb: torch.Size([8, 50])
train: 14 | eval: 6


## 3. Hyperparameter Search

We define a grid of hyperparameters (embedding dimension, number of attention heads, transformer layers, dropout rate, epochs, batch size, learning rate) and train a separate model for each combination. For each configuration we:
- Instantiate the TinyGPT model.
- Train it on the training set.
- Save model weights and generate Arduino-compatible C++ code.
- Evaluate on train and test sets.
- Collect metrics (loss, perplexity, token accuracy, BLEU, exact match) and model size information.

All results are saved to JSON and CSV files for later analysis.

In [6]:

import itertools
import json
import os
import csv
from typing import Dict, List, Any


# ============================================================
# DATASET
# ============================================================

# Número de exemplos utilizados no experimento.
#
# Exemplo:
# n = 20
#
# Para utilizar até 5000 exemplos:
# n = min(5000, len(filtered_qna))

n = 20


# ------------------------------------------------------------
# Seleciona os primeiros n exemplos
# ------------------------------------------------------------

train_qna_full = filtered_qna[:n]


# ------------------------------------------------------------
# Train / Evaluation split
# ------------------------------------------------------------

split = int(len(train_qna_full) * 0.7)

train_qna = train_qna_full[:split]
eval_qna = train_qna_full[split:]


# ------------------------------------------------------------
# Informações do dataset
# ------------------------------------------------------------

print("=" * 60)
print("DATASET")
print("=" * 60)

print("Total:", len(train_qna_full))
print("Train:", len(train_qna))
print("Eval :", len(eval_qna))

print("=" * 60)


# ============================================================
# TOKENIZER
# ============================================================

# O tokenizer é construído usando TODO o filtered_qna,
# conforme solicitado.
#
# Isso garante que train e eval utilizem o mesmo vocabulário.

tokenizer = CustomTokenizer(
    data_source=filtered_qna
)


# ============================================================
# CONFIGURAÇÃO
# ============================================================

MAX_LEN = 50


print("vocab_size:", tokenizer.vocab_size)
print("MAX_LEN:", MAX_LEN)


# ============================================================
# DATASET / DATALOADER
# ============================================================
#
# Estes objetos são mantidos apenas para inspeção/uso externo.
#
# IMPORTANTE:
#
# TinyGPT.train() recebe a LISTA de Q&A:
#
#     train_qna
#
# e não:
#
#     train_dataset
#
# Isso ocorre porque TinyGPT cria internamente o tokenizer e
# o DataLoader.

train_dataset = QADataset(
    qna_list=train_qna,
    tokenizer=tokenizer,
    max_length=MAX_LEN
)


eval_dataset = QADataset(
    qna_list=eval_qna,
    tokenizer=tokenizer,
    max_length=MAX_LEN
)


loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True
)


eval_loader = DataLoader(
    eval_dataset,
    batch_size=8,
    shuffle=False
)


# ------------------------------------------------------------
# Verificação dos dados
# ------------------------------------------------------------

if len(train_dataset) > 0:

    xb, yb = next(iter(loader))

    print(
        "vocab_size:",
        tokenizer.vocab_size,
        "| xb:",
        xb.shape,
        "| yb:",
        yb.shape
    )


topics = [""] * len(eval_qna)


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def flatten_json_to_list(
    input_json_path: str
) -> List[float]:

    """
    Extracts all numeric values from a JSON file recursively.
    """

    with open(
        input_json_path,
        "r"
    ) as f:

        json_data = json.load(f)

    values = []


    def extract_values(data):

        if isinstance(data, dict):

            for key, value in data.items():

                extract_values(value)


        elif isinstance(data, list):

            for item in data:

                if isinstance(item, list):

                    for num in item:

                        if isinstance(
                            num,
                            (int, float)
                        ):

                            values.append(num)


                elif isinstance(
                    item,
                    (int, float)
                ):

                    values.append(item)


        elif isinstance(
            data,
            (int, float)
        ):

            values.append(data)


        elif data is None:

            pass


        else:

            raise ValueError(
                f"Unsupported type: {type(data)}"
            )


    extract_values(json_data)

    return values


# ============================================================

def count_total_parameters(
    path_input: str
) -> int:

    """
    Returns the total number of numeric parameters by summing:

    - embedding.json
    - final_norm.json
    - out_head.json
    - transformer_blocks.json
    """

    file_names = [

        "embedding.json",

        "final_norm.json",

        "out_head.json",

        "transformer_blocks.json"

    ]


    total_params = 0


    for file_name in file_names:

        file_path = os.path.join(
            path_input,
            file_name
        )


        try:

            flattened = flatten_json_to_list(
                input_json_path=file_path
            )

            total_params += len(flattened)


        except FileNotFoundError:

            print(
                f"Warning: file not found: {file_name}"
            )


        except Exception as e:

            print(
                f"Error processing {file_name}: {e}"
            )


    return total_params


# ============================================================

def get_total_cpp_header_size(
    folder_path: str
) -> int:

    """
    Returns the total size in bytes of .cpp and .h
    files in the specified folder.
    """

    total_size = 0


    if not os.path.exists(folder_path):

        print(
            f"Warning: directory not found: {folder_path}"
        )

        return 0


    for file_name in os.listdir(folder_path):

        if file_name.endswith(
            (".cpp", ".h")
        ):

            file_path = os.path.join(
                folder_path,
                file_name
            )

            total_size += os.path.getsize(
                file_path
            )


    return total_size


# ============================================================
# HYPERPARAMETER GRID
# ============================================================

param_grid = {

    "embedding_dim": [
        12
    ],

    "heads": [
        2
    ],

    "layers": [
        1
    ],

    "drop_rate": [
        0.01
    ],

    "num_epochs": [
        300
    ],

    "batch_size": [
        4
    ],

    "lr": [
        0.01
    ]

}


# ============================================================
# GENERATE COMBINATIONS
# ============================================================

keys = param_grid.keys()

values = param_grid.values()

combinations = list(
    itertools.product(
        *values
    )
)


print()
print("=" * 60)
print("HYPERPARAMETER SEARCH")
print("=" * 60)

print(
    "Total configurations:",
    len(combinations)
)

print("=" * 60)


# ============================================================
# RESULTS
# ============================================================

all_results: List[
    Dict[str, Any]
] = []


# ============================================================
# MAIN LOOP
# ============================================================

for idx, combo in enumerate(
    combinations
):


    # --------------------------------------------------------
    # Configuration dictionary
    # --------------------------------------------------------

    params = dict(
        zip(
            keys,
            combo
        )
    )


    # --------------------------------------------------------
    # Configuration ID
    # --------------------------------------------------------

    config_id = (

        f"emb{params['embedding_dim']}_"

        f"h{params['heads']}_"

        f"l{params['layers']}_"

        f"d{str(params['drop_rate']).replace('.', '')}_"

        f"ep{params['num_epochs']}_"

        f"bs{params['batch_size']}_"

        f"lr{str(params['lr']).replace('.', '')}"

    )


    # --------------------------------------------------------
    # Console
    # --------------------------------------------------------

    print()
    print("=" * 60)

    print(
        f"Running configuration "
        f"{idx + 1}/{len(combinations)}"
    )

    print(
        f"Configuration ID: {config_id}"
    )

    print(
        f"Parameters: {params}"
    )

    print("=" * 60)


    # ========================================================
    # OUTPUT DIRECTORIES
    # ========================================================

    output_model_dir = (
        f"output_dir_{config_id}"
    )

    output_arduino_dir = (
        f"output_arduino_{config_id}"
    )


    os.makedirs(
        output_model_dir,
        exist_ok=True
    )


    os.makedirs(
        output_arduino_dir,
        exist_ok=True
    )


    # ========================================================
    # MODEL
    # ========================================================

    model = TinyGPT(

        embedding_dim=params[
            "embedding_dim"
        ],

        heads=params[
            "heads"
        ],

        layers=params[
            "layers"
        ],

        drop_rate=params[
            "drop_rate"
        ]

    )


    # ========================================================
    # TRAINING
    # ========================================================
    #
    # IMPORTANTE:
    #
    # NÃO usar:
    #
    #     synthetic_dataset=train_dataset
    #
    # pois train_dataset é um QADataset.
    #
    # TinyGPT espera uma lista de dicionários Q&A.
    #
    # Portanto:
    #
    #     synthetic_dataset=train_qna

    print()
    print("Training...")

    model.train(

        synthetic_dataset=train_qna,

        num_epochs=params[
            "num_epochs"
        ],

        batch_size=params[
            "batch_size"
        ],

        lr=params[
            "lr"
        ],

        optimizer_cls=AdamW,

        loss_fn_cls=CrossEntropyLoss

    )


    # ========================================================
    # SAVE MODEL
    # ========================================================

    model.save(
        output_folder=output_model_dir
    )


    # ========================================================
    # GENERATE ARDUINO CODE
    # ========================================================

    model.code_generator(

        model_folder=output_model_dir,

        output_arduino_folder=output_arduino_dir,

        model_name=f"model_{config_id}"

    )


    # ========================================================
    # TRAIN EVALUATION
    # ========================================================
    #
    # Novamente usamos train_qna, e não train_dataset.

    print()
    print("Evaluating train set...")

    train_metrics = model.evaluation(

        train_qna,

        batch_size=32,

        num_examples=len(
            train_qna
        )

    )


    # ========================================================
    # EVAL EVALUATION
    # ========================================================
    #
    # Avaliação em exemplos que NÃO foram utilizados
    # durante o treinamento.

    print("Evaluating eval set...")

    eval_metrics = model.evaluation(

        eval_qna,

        batch_size=32,

        num_examples=len(
            eval_qna
        )

    )


    # ========================================================
    # SAVE TRAIN METRICS
    # ========================================================

    train_json_path = (
        f"evaluation_train_{config_id}.json"
    )


    with open(
        train_json_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(

            train_metrics,

            f,

            indent=2,

            ensure_ascii=False

        )


    # ========================================================
    # SAVE EVAL METRICS
    # ========================================================

    eval_json_path = (
        f"evaluation_eval_{config_id}.json"
    )


    with open(
        eval_json_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(

            eval_metrics,

            f,

            indent=2,

            ensure_ascii=False

        )


    # ========================================================
    # COUNT PARAMETERS
    # ========================================================

    num_params = count_total_parameters(
        output_model_dir
    )


    # ========================================================
    # C++ SIZE
    # ========================================================

    cpp_size = get_total_cpp_header_size(
        output_arduino_dir
    )


    # ========================================================
    # STORE RESULT
    # ========================================================

    result_entry = {

        "config_id":
            config_id,

        "params":
            params,

        "train_metrics":
            train_metrics,

        "eval_metrics":
            eval_metrics,

        "train_json":
            train_json_path,

        "eval_json":
            eval_json_path,

        "model_dir":
            output_model_dir,

        "arduino_dir":
            output_arduino_dir,

        "number_parameter":
            num_params,

        "cpp_model_size":
            cpp_size

    }


    all_results.append(
        result_entry
    )


    # ========================================================
    # CONSOLE METRICS
    # ========================================================

    train_loss = train_metrics.get(
        "avg_loss",
        float("nan")
    )

    train_acc = train_metrics.get(
        "token_accuracy",
        float("nan")
    )


    eval_loss = eval_metrics.get(
        "avg_loss",
        float("nan")
    )

    eval_acc = eval_metrics.get(
        "token_accuracy",
        float("nan")
    )


    print()
    print("-" * 60)

    print(
        f"Train - Avg loss: {train_loss:.4f}"
    )

    print(
        f"Train - Token accuracy: {train_acc:.4f}"
    )

    print(
        f"Eval  - Avg loss: {eval_loss:.4f}"
    )

    print(
        f"Eval  - Token accuracy: {eval_acc:.4f}"
    )

    print(
        f"Model parameters: {num_params}"
    )

    print(
        f"C++ code size: {cpp_size} bytes"
    )

    print("-" * 60)


# ============================================================
# SAVE OVERALL JSON SUMMARY
# ============================================================

summary_path = (
    "hyperparameter_search_summary.json"
)


with open(
    summary_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(

        all_results,

        f,

        indent=2,

        ensure_ascii=False

    )


print()
print("=" * 60)

print(
    "Hyperparameter search completed!"
)

print(
    f"Results saved to '{summary_path}'."
)

print("=" * 60)


# ============================================================
# CSV
# ============================================================

csv_path = (
    "hyperparameter_search_summary.csv"
)


fieldnames = [

    "config_id",

    "embedding_dim",
    "heads",
    "layers",
    "drop_rate",

    "num_epochs",
    "batch_size",
    "lr",

    # --------------------------------------------------------
    # Train metrics
    # --------------------------------------------------------

    "train_avg_loss",
    "train_perplexity",
    "train_token_accuracy",
    "train_num_samples",
    "train_avg_bleu",
    "train_exact_match_rate",

    # --------------------------------------------------------
    # Eval metrics
    # --------------------------------------------------------

    "eval_avg_loss",
    "eval_perplexity",
    "eval_token_accuracy",
    "eval_num_samples",
    "eval_avg_bleu",
    "eval_exact_match_rate",

    # --------------------------------------------------------
    # Model
    # --------------------------------------------------------

    "number_parameter",
    "cpp_model_size"

]


with open(
    csv_path,
    "w",
    newline="",
    encoding="utf-8"
) as csvfile:

    writer = csv.DictWriter(
        csvfile,
        fieldnames=fieldnames
    )

    writer.writeheader()


    # ========================================================
    # WRITE RESULTS
    # ========================================================

    for entry in all_results:

        train = entry[
            "train_metrics"
        ]

        evaluation = entry[
            "eval_metrics"
        ]


        row = {

            # ------------------------------------------------
            # Configuration
            # ------------------------------------------------

            "config_id":
                entry["config_id"],

            "embedding_dim":
                entry["params"][
                    "embedding_dim"
                ],

            "heads":
                entry["params"][
                    "heads"
                ],

            "layers":
                entry["params"][
                    "layers"
                ],

            "drop_rate":
                entry["params"][
                    "drop_rate"
                ],

            "num_epochs":
                entry["params"][
                    "num_epochs"
                ],

            "batch_size":
                entry["params"][
                    "batch_size"
                ],

            "lr":
                entry["params"][
                    "lr"
                ],


            # ------------------------------------------------
            # Train
            # ------------------------------------------------

            "train_avg_loss":
                train.get(
                    "avg_loss",
                    ""
                ),

            "train_perplexity":
                train.get(
                    "perplexity",
                    ""
                ),

            "train_token_accuracy":
                train.get(
                    "token_accuracy",
                    ""
                ),

            "train_num_samples":
                train.get(
                    "num_samples",
                    ""
                ),

            "train_avg_bleu":
                train.get(
                    "avg_bleu",
                    ""
                ),

            "train_exact_match_rate":
                train.get(
                    "exact_match_rate",
                    ""
                ),


            # ------------------------------------------------
            # Eval
            # ------------------------------------------------

            "eval_avg_loss":
                evaluation.get(
                    "avg_loss",
                    ""
                ),

            "eval_perplexity":
                evaluation.get(
                    "perplexity",
                    ""
                ),

            "eval_token_accuracy":
                evaluation.get(
                    "token_accuracy",
                    ""
                ),

            "eval_num_samples":
                evaluation.get(
                    "num_samples",
                    ""
                ),

            "eval_avg_bleu":
                evaluation.get(
                    "avg_bleu",
                    ""
                ),

            "eval_exact_match_rate":
                evaluation.get(
                    "exact_match_rate",
                    ""
                ),


            # ------------------------------------------------
            # Model
            # ------------------------------------------------

            "number_parameter":
                entry[
                    "number_parameter"
                ],

            "cpp_model_size":
                entry[
                    "cpp_model_size"
                ]

        }


        writer.writerow(row)


print(
    f"Tabular summary saved to '{csv_path}'."
)

DATASET
Total: 20
Train: 14
Eval : 6
[INFO] Vocabulary built from Q&A with 489 tokens.
vocab_size: 489
MAX_LEN: 50
[INFO] Max original sequence length found: 22
[INFO] Sequences adjusted to max_length: 50
[INFO] Max original sequence length found: 20
[INFO] Sequences adjusted to max_length: 50
vocab_size: 489 | xb: torch.Size([8, 50]) | yb: torch.Size([8, 50])

HYPERPARAMETER SEARCH
Total configurations: 1

Running configuration 1/1
Configuration ID: emb12_h2_l1_d001_ep300_bs4_lr001
Parameters: {'embedding_dim': 12, 'heads': 2, 'layers': 1, 'drop_rate': 0.01, 'num_epochs': 300, 'batch_size': 4, 'lr': 0.01}

Training...
[INFO] Vocabulary built from Q&A with 112 tokens.
[INFO] Context length: 24
[INFO] Max original sequence length found: 22
[INFO] Sequences adjusted to max_length: 24
[INFO] Using Apple MPS (Metal Performance Shaders).
                  TRAINING MODEL                  
[INFO] Epoch 1/300 - Loss: 4.8911
[INFO] Epoch 2/300 - Loss: 4.4828
[INFO] Epoch 3/300 - Loss: 4.1653
[I